In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ติดตามหรือยกเลิก job

ดูรายการ workload ของคุณได้ที่ [Workloads page](https://quantum.cloud.ibm.com/workloads)

## ดูสถานะ job
ไปที่ [Workloads table](https://quantum.cloud.ibm.com/workloads) และตรวจสอบใต้คอลัมน์ Status เพื่อดูว่า job เสร็จสมบูรณ์หรือล้มเหลว

## ดูการใช้งานที่เหลือ
ไปที่ [Instances table](https://quantum.cloud.ibm.com/instances) และเลือกแท็บที่เกี่ยวข้องกับแผนที่คุณต้องการดูการใช้งานที่เหลือ จะแสดงเวลาทั้งหมดที่ใช้ไปและเวลาทั้งหมดที่เหลือในแผนของคุณ

## ดูตัวชี้วัดจำนวน job และ workload ที่ส่ง
ไปที่ [Analytics page](https://quantum.cloud.ibm.com/analytics) เพื่อดูจำนวน job ทั้งหมดที่ส่ง รวมถึงจำนวน batch workload และ session workload โปรดทราบว่าคุณสามารถดู Analytics page ได้เฉพาะสำหรับบัญชีที่คุณเป็นเจ้าของหรือจัดการ

## ติดตาม job
ใช้ job instance เพื่อตรวจสอบสถานะ job หรือดึงผลลัพธ์โดยเรียกคำสั่งที่เหมาะสม:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | ดูผลลัพธ์ job ทันทีหลังจาก job เสร็จสมบูรณ์ ผลลัพธ์ job พร้อมใช้งานหลังจาก job เสร็จสมบูรณ์ ดังนั้น job.result() คือการเรียกแบบ blocking จนกว่า job จะเสร็จสมบูรณ์                               |
| job.job\_id()                 | คืนค่า ID ที่ระบุ job นั้นได้อย่างเป็นเอกลักษณ์ การดึงผลลัพธ์ job ในภายหลังต้องใช้ job ID ดังนั้นแนะนำให้บันทึก ID ของ job ที่คุณอาจต้องการดึงในภายหลัง |
| job.status()                  | ตรวจสอบสถานะ job                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | ดึง job ที่คุณเคยส่งไว้ก่อนหน้า การเรียกนี้ต้องใช้ job ID                                                                                                                                                      |

<span id="retrieve-later"></span>
## ดึงผลลัพธ์ job ในภายหลัง
เรียก `service.job(\<job\_id>)` เพื่อดึง job ที่คุณเคยส่งไว้ก่อนหน้า ถ้าคุณไม่มี job ID หรือถ้าคุณต้องการดึง job หลายตัวพร้อมกัน รวมถึง job จาก QPU (quantum processing unit) ที่เลิกใช้งานแล้ว ให้เรียก `service.jobs()` พร้อม filter ที่ต้องการแทน ดู [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs)

> **Note:** `service.jobs()` ยังคืนค่า job ที่รันจาก `qiskit-ibm-provider` package ที่ถูก deprecated แล้วด้วย Job ที่ส่งโดย `qiskit-ibmq-provider` package รุ่นเก่า (ที่ deprecated แล้วเช่นกัน) ไม่สามารถใช้งานได้อีกต่อไป

### ตัวอย่าง
ตัวอย่างนี้คืนค่า 10 runtime job ล่าสุดที่รันบน `my_backend`:

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>